# Hello World — Spark + Delta Lake + MinIO (s3a)

Notebook de validação da ISSUE #21. Demonstra:
- SparkSession configurada para MinIO via `s3a://`
- Delta Lake habilitado
- Leitura da camada Landing
- Escrita Delta na camada Bronze (`demo/`)

In [1]:
# Célula 0 — Instalar dependências ausentes no container
import subprocess
import importlib.util

pkgs = {
    "delta": "delta-spark==3.2.0",
    "minio": "minio==7.2.7",
    "dotenv": "python-dotenv",
}
for module, pkg in pkgs.items():
    if importlib.util.find_spec(module) is None:
        print(f"Instalando {pkg} ...")
        subprocess.run(["pip", "install", "--quiet", pkg], check=True)
        print(f"{pkg} instalado.")
    else:
        print(f"{module} ja disponivel.")

delta ja disponivel.
minio ja disponivel.
dotenv ja disponivel.


In [2]:
# Célula 1 — Setup: sys.path e variáveis de ambiente
import os
import sys

# Garante que src/ está no path para importar utils.spark_config
project_root = "/home/jovyan/work"
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Carrega variáveis de ambiente do .env (fallback: já devem estar no container)
try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(project_root, ".env"))
    print("Arquivo .env carregado.")
except ImportError:
    print("python-dotenv nao instalado — usando variaveis ja presentes no ambiente.")

print(f"MINIO_ROOT_USER  = {os.getenv('MINIO_ROOT_USER', '(nao definido)')}")
print(f"MINIO_ENDPOINT   = {os.getenv('MINIO_ENDPOINT', 'minio:9000 (padrao)')}")

Arquivo .env carregado.
MINIO_ROOT_USER  = admin
MINIO_ENDPOINT   = minio:9000


In [3]:
# Célula 2 — Criar SparkSession com Delta + s3a
from utils.spark_config import build_spark_session

spark = build_spark_session(app_name="hello-world-delta-minio")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print("SparkSession criada com sucesso.")

Spark version : 3.5.3
App name      : hello-world-delta-minio
SparkSession criada com sucesso.


In [4]:
# Célula 3 — Verificar conectividade MinIO e listar buckets
from minio import Minio

minio_client = Minio(
    endpoint=os.getenv("MINIO_ENDPOINT", "minio:9000"),
    access_key=os.getenv("MINIO_ROOT_USER"),
    secret_key=os.getenv("MINIO_ROOT_PASSWORD"),
    secure=False,
)

print("=== Status dos buckets ===")
buckets = ["landing", "bronze", "silver", "gold"]
for b in buckets:
    exists = minio_client.bucket_exists(b)
    status = "OK" if exists else "NAO ENCONTRADO"
    print(f"  {b:<10} {status}")

landing_objects = list(minio_client.list_objects("landing", recursive=True))
print(f"\nObjetos na landing: {len(landing_objects)}")
for obj in landing_objects[:10]:
    print(f"  {obj.object_name}  ({obj.size} bytes)")

=== Status dos buckets ===
  landing    OK
  bronze     OK
  silver     OK
  gold       OK

Objetos na landing: 0


In [5]:
# Célula 4 — Ler da Landing (tolerante a bucket vazio)
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

LANDING_PATH = "s3a://landing/"

if not landing_objects:
    print("Landing bucket vazio — criando DataFrame de exemplo para demo.")
    # Schema espelhando a tabela 'streamers' do PostgreSQL
    schema = StructType([
        StructField("id_streamer",   IntegerType(), False),
        StructField("nome",          StringType(),  True),
        StructField("pais",          StringType(),  True),
        StructField("data_cadastro", StringType(),  True),
        StructField("id_plataforma", IntegerType(), True),
    ])
    sample_data = [
        (1, "StreamerExemplo", "Brasil", "2024-01-01", 1),
        (2, "AnotherStreamer", "EUA",    "2024-03-15", 2),
        (3, "GamerPT",        "Portugal","2024-06-10", 1),
    ]
    df_landing = spark.createDataFrame(sample_data, schema=schema)
    print("DataFrame de exemplo criado.")
else:
    print(f"Lendo CSVs de {LANDING_PATH} ...")
    df_landing = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(LANDING_PATH)
    )
    print(f"Registros lidos: {df_landing.count()}")

df_landing.printSchema()
df_landing.show(5, truncate=False)

Landing bucket vazio — criando DataFrame de exemplo para demo.
DataFrame de exemplo criado.
root
 |-- id_streamer: integer (nullable = false)
 |-- nome: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- data_cadastro: string (nullable = true)
 |-- id_plataforma: integer (nullable = true)

+-----------+---------------+--------+-------------+-------------+
|id_streamer|nome           |pais    |data_cadastro|id_plataforma|
+-----------+---------------+--------+-------------+-------------+
|1          |StreamerExemplo|Brasil  |2024-01-01   |1            |
|2          |AnotherStreamer|EUA     |2024-03-15   |2            |
|3          |GamerPT        |Portugal|2024-06-10   |1            |
+-----------+---------------+--------+-------------+-------------+



In [6]:
# Célula 5 — Escrever Delta Lake na camada Bronze (pasta demo/)
BRONZE_PATH = "s3a://bronze/demo/streamers_hello_world"

print(f"Escrevendo Delta Lake em {BRONZE_PATH} ...")
(
    df_landing.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BRONZE_PATH)
)
print("Escrita Delta concluida.")

# Lê de volta para confirmar
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
print(f"Registros lidos do bronze: {df_bronze.count()}")
df_bronze.show(5, truncate=False)

Escrevendo Delta Lake em s3a://bronze/demo/streamers_hello_world ...
Escrita Delta concluida.
Registros lidos do bronze: 3
+-----------+---------------+--------+-------------+-------------+
|id_streamer|nome           |pais    |data_cadastro|id_plataforma|
+-----------+---------------+--------+-------------+-------------+
|1          |StreamerExemplo|Brasil  |2024-01-01   |1            |
|2          |AnotherStreamer|EUA     |2024-03-15   |2            |
|3          |GamerPT        |Portugal|2024-06-10   |1            |
+-----------+---------------+--------+-------------+-------------+



In [7]:
# Célula 6 — Inspecionar histórico Delta e encerrar sessão
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, BRONZE_PATH)

print("=== Delta Lake History ===")
dt.history().select("version", "timestamp", "operation", "operationParameters").show(
    truncate=False
)

print("\n=== Delta Lake Detail ===")
dt.detail().select("name", "numFiles", "sizeInBytes", "format").show(truncate=False)

spark.stop()
print("\nSparkSession encerrada. Hello World concluido com sucesso!")

=== Delta Lake History ===
+-------+-------------------+---------+--------------------------------------+
|version|timestamp          |operation|operationParameters                   |
+-------+-------------------+---------+--------------------------------------+
|0      |2026-06-22 18:03:42|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-------------------+---------+--------------------------------------+


=== Delta Lake Detail ===
+----+--------+-----------+------+
|name|numFiles|sizeInBytes|format|
+----+--------+-----------+------+
|NULL|3       |4710       |delta |
+----+--------+-----------+------+


SparkSession encerrada. Hello World concluido com sucesso!


In [8]:
import pyspark
print(pyspark.__version__)

3.5.3


In [9]:
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)



openjdk version "17.0.12" 2024-07-16
OpenJDK Runtime Environment (build 17.0.12+7-Ubuntu-1ubuntu224.04)
OpenJDK 64-Bit Server VM (build 17.0.12+7-Ubuntu-1ubuntu224.04, mixed mode, sharing)



In [10]:
import subprocess
result = subprocess.run(
    ["find", "/usr/local/spark", "-name", "*.jar", "-path", "*/assembly/*"],
    capture_output=True, text=True
)
print(result.stdout[:2000])

In [11]:
import subprocess
result = subprocess.run(["find", "/opt/conda", "-name", "spark-submit", "-o", "-name", "spark-class"], capture_output=True, text=True)
print(result.stdout)